# 조인 차이 appid / 게임명 확인

1. `steam_indie_tags`에는 있는데 `steam_indie_games`에는 없는 113개 게임 확인
2. 리뷰 관련 데이터에는 있는데 `steam_indie_games`에는 없는 3개 게임 확인
3. `review_summary`, `review_histogram`은 200개인데 `reviews`는 197개인 이유 확인
4. 차이가 나는 `appid`와 게임명을 표로 확인

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
from IPython.display import display

# 보기 옵션
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

In [3]:
import sys
from pathlib import Path

import pandas as pd

# 프로젝트 루트 경로 등록
sys.path.insert(0, str(Path("../../..").resolve()))

from src.utils.db import get_connection


conn = get_connection()

try:
    games_df = pd.read_sql("SELECT * FROM steam_indie_games", conn)
    reviews_df = pd.read_sql("SELECT * FROM steam_indie_reviews", conn)
    review_summary_df = pd.read_sql("SELECT * FROM steam_indie_review_summary", conn)
    review_histogram_df = pd.read_sql("SELECT * FROM steam_indie_review_histogram", conn)
    tags_df = pd.read_sql("SELECT * FROM steam_indie_tags", conn)

finally:
    conn.close()

C:\Users\joon5\AppData\Local\Temp\ipykernel_26868\358496381.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  games_df = pd.read_sql("SELECT * FROM steam_indie_games", conn)
C:\Users\joon5\AppData\Local\Temp\ipykernel_26868\358496381.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  reviews_df = pd.read_sql("SELECT * FROM steam_indie_reviews", conn)
C:\Users\joon5\AppData\Local\Temp\ipykernel_26868\358496381.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  review_summary_df = pd.read_sql("SELECT * FROM steam_indie

In [4]:
print("games_df shape            :", games_df.shape)
print("tags_df shape             :", tags_df.shape)
print("summary_df shape          :", review_summary_df.shape)
print("histogram_df shape        :", review_histogram_df.shape)
print("reviews_df shape          :", reviews_df.shape)

games_df shape            : (9692, 14)
tags_df shape             : (9706, 9)
summary_df shape          : (200, 6)
histogram_df shape        : (11782, 10)
reviews_df shape          : (236379, 21)


## 2. appid 타입 통일

In [5]:
# 조인 비교를 하기 전에 appid 타입을 숫자형으로 통일한다.
# CSV마다 appid 타입이 다르면 같은 값이어도 매칭이 안 될 수 있기 때문이다.
tables = {
    "steam_indie_games": games_df,
    "steam_indie_tags": tags_df,
    "steam_indie_review_summary": review_summary_df,
    "steam_indie_review_histogram": review_histogram_df,
    "steam_indie_reviews": reviews_df,
}

for table_name, df in tables.items():
    df["appid"] = pd.to_numeric(df["appid"], errors="coerce").astype("Int64")

    na_count = df["appid"].isna().sum()
    print(f"{table_name:30s} appid 결측/변환 실패 수: {na_count}")

steam_indie_games              appid 결측/변환 실패 수: 0
steam_indie_tags               appid 결측/변환 실패 수: 0
steam_indie_review_summary     appid 결측/변환 실패 수: 0
steam_indie_review_histogram   appid 결측/변환 실패 수: 0
steam_indie_reviews            appid 결측/변환 실패 수: 0


# 3. 전처리 전 기본 확인

In [6]:
def get_unique_appids(df):
    """DataFrame에서 결측을 제외한 고유 appid 집합을 반환한다."""
    return set(df["appid"].dropna().astype(int).unique())


def check_join_summary(base_df, target_df, base_name, target_name, key="appid"):
    """
    기준 테이블의 appid가 대상 테이블에 얼마나 존재하는지 확인한다.

    - 기준 appid 수: 기준 테이블의 고유 appid 개수
    - 매칭 appid 수: 기준과 대상에 모두 존재하는 appid 개수
    - 미매칭 appid 수: 기준에는 있지만 대상에는 없는 appid 개수
    - 대상에만 있는 appid 수: 대상에는 있지만 기준에는 없는 appid 개수
    """
    base_ids = set(base_df[key].dropna().astype(int).unique())
    target_ids = set(target_df[key].dropna().astype(int).unique())

    matched_ids = base_ids & target_ids
    base_only_ids = base_ids - target_ids
    target_only_ids = target_ids - base_ids

    return {
        "기준 테이블": base_name,
        "대상 테이블": target_name,
        "기준 appid 수": len(base_ids),
        "매칭 appid 수": len(matched_ids),
        "미매칭 appid 수": len(base_only_ids),
        "매칭률": round(len(matched_ids) / len(base_ids) * 100, 2) if len(base_ids) > 0 else np.nan,
        "대상에만 있는 appid 수": len(target_only_ids),
    }

# 조인 가능성 표 다시 확인

1. 게임 ↔ 태그 매칭 확인
2. 리뷰 데이터 → 게임 데이터 매칭 확인
3. 게임 데이터 → 리뷰 데이터 매칭 확인

In [7]:
# 분석 흐름상 먼저 확인할 순서로 조인 가능성 표를 만든다.
join_checks = [
    # 게임-태그는 게임 단위 기본 속성 데이터끼리의 관계이므로 먼저 확인한다.
    check_join_summary(games_df, tags_df, "steam_indie_games", "steam_indie_tags"),
    check_join_summary(tags_df, games_df, "steam_indie_tags", "steam_indie_games"),

    # 리뷰 관련 데이터는 표본 데이터이므로 리뷰 데이터 → 게임 데이터 방향을 먼저 확인한다.
    check_join_summary(review_summary_df, games_df, "steam_indie_review_summary", "steam_indie_games"),
    check_join_summary(review_histogram_df, games_df, "steam_indie_review_histogram", "steam_indie_games"),
    check_join_summary(reviews_df, games_df, "steam_indie_reviews", "steam_indie_games"),

    # 반대로 전체 게임 데이터 기준에서 리뷰 데이터가 얼마나 붙는지도 참고용으로 확인한다.
    check_join_summary(games_df, review_summary_df, "steam_indie_games", "steam_indie_review_summary"),
    check_join_summary(games_df, review_histogram_df, "steam_indie_games", "steam_indie_review_histogram"),
    check_join_summary(games_df, reviews_df, "steam_indie_games", "steam_indie_reviews"),
]

join_summary_df = pd.DataFrame(join_checks)
display(join_summary_df)

,기준 테이블,대상 테이블,기준 appid 수,매칭 appid 수,미매칭 appid 수,매칭률,대상에만 있는 appid 수
0,steam_indie_games,steam_indie_tags,9692,9692,0,100.00,14
1,steam_indie_tags,steam_indie_games,9706,9692,14,99.86,0
2,steam_indie_review_summary,steam_indie_games,200,200,0,100.00,9492
3,steam_indie_review_histogram,steam_indie_games,200,200,0,100.00,9492
4,steam_indie_reviews,steam_indie_games,197,197,0,100.00,9495
5,steam_indie_games,steam_indie_review_summary,9692,200,9492,2.06,0
6,steam_indie_games,steam_indie_review_histogram,9692,200,9492,2.06,0
7,steam_indie_games,steam_indie_reviews,9692,197,9495,2.03,0


## 4. appid 기준 통합 참조표

In [8]:
# 각 테이블의 고유 appid 집합을 만든다.
appid_sets = {
    "games": get_unique_appids(games_df),
    "tags": get_unique_appids(tags_df),
    "review_summary": get_unique_appids(review_summary_df),
    "review_histogram": get_unique_appids(review_histogram_df),
    "reviews": get_unique_appids(reviews_df),
}

# 전체 appid 목록을 하나로 합친다.
all_appids = sorted(set().union(*appid_sets.values()))
appid_ref_df = pd.DataFrame({"appid": all_appids})

# 각 테이블의 이름 컬럼을 appid 기준으로 붙인다.
# games, tags, histogram은 name 컬럼을 가지고 있다.
name_sources = [
    ("games", games_df, "name"),
    ("tags", tags_df, "name"),
    ("review_histogram", review_histogram_df, "name"),
]

for source_name, source_df, name_col in name_sources:
    if name_col in source_df.columns:
        temp = (
            source_df[["appid", name_col]]
            .dropna(subset=["appid"])
            .drop_duplicates("appid")
            .rename(columns={name_col: f"name_{source_name}"})
        )
        appid_ref_df = appid_ref_df.merge(temp, on="appid", how="left")

# 여러 이름 컬럼 중 앞에서부터 값이 있는 것을 대표 게임명으로 사용한다.
appid_ref_df["game_name"] = (
    appid_ref_df.get("name_games")
    .combine_first(appid_ref_df.get("name_tags"))
    .combine_first(appid_ref_df.get("name_review_histogram"))
)

# 각 appid가 어느 테이블에 존재하는지 표시한다.
for source_name, ids in appid_sets.items():
    appid_ref_df[f"in_{source_name}"] = appid_ref_df["appid"].isin(ids)

# 보기 좋은 컬럼 순서로 정리한다.
ref_cols = [
    "appid", "game_name", "name_games", "name_tags", "name_review_histogram",
    "in_games", "in_tags", "in_review_summary", "in_review_histogram", "in_reviews"
]
appid_ref_df = appid_ref_df[ref_cols]

display(appid_ref_df.head())

,appid,game_name,name_games,name_tags,name_review_histogram,in_games,in_tags,in_review_summary,in_review_histogram,in_reviews
0,226620,Desktop Dungeons,Desktop Dungeons,Desktop Dungeons,NaN,True,True,False,False,False
1,230210,ASYLUM,ASYLUM,ASYLUM,NaN,True,True,False,False,False
2,251570,7 Days to Die,7 Days to Die,7 Days to Die,NaN,True,True,False,False,False
3,252190,Defender's Quest 2: Mists of Ruin,Defender's Quest 2: Mists of Ruin,Defender's Quest 2: Mists of Ruin,NaN,True,True,False,False,False
4,269770,Secrets of Grindea,Secrets of Grindea,Secrets of Grindea,NaN,True,True,False,False,False


## 5. 차이가 나는 appid와 게임명 확인

In [9]:
def show_appid_difference(base_name, target_name, title=None):
    """
    base_name에는 있지만 target_name에는 없는 appid를 게임명과 함께 보여준다.

    예:
    - show_appid_difference("tags", "games")
      → tags에는 있는데 games에는 없는 게임 확인
    """
    base_ids = appid_sets[base_name]
    target_ids = appid_sets[target_name]
    diff_ids = sorted(base_ids - target_ids)

    result = (
        appid_ref_df[appid_ref_df["appid"].isin(diff_ids)]
        .sort_values("appid")
        .reset_index(drop=True)
    )

    if title is not None:
        print(title)
    print(f"{base_name}에는 있지만 {target_name}에는 없는 appid 수: {len(result)}")
    display(result)

    return result

## tags에는 있지만 games에는 없는 113개 확인

In [10]:
tags_only_not_games_df = show_appid_difference(
    "tags",
    "games",
    title="steam_indie_tags에는 있지만 steam_indie_games에는 없는 게임"
)

steam_indie_tags에는 있지만 steam_indie_games에는 없는 게임
tags에는 있지만 games에는 없는 appid 수: 14


,appid,game_name,name_games,name_tags,name_review_histogram,in_games,in_tags,in_review_summary,in_review_histogram,in_reviews
0,1049590,Eternal Return,NaN,Eternal Return,NaN,False,True,False,False,False
1,1116540,DAVIGO: VR vs. PC,NaN,DAVIGO: VR vs. PC,NaN,False,True,False,False,False
2,1213300,Rebellion GODSOUL: Awakening,NaN,Rebellion GODSOUL: Awakening,NaN,False,True,False,False,False
3,1623730,Palworld,NaN,Palworld,NaN,False,True,False,False,False
4,1700300,World Warfare & Economics,NaN,World Warfare & Economics,NaN,False,True,False,False,False
5,1948800,Yi Xian: The Cultivation Card Game,NaN,Yi Xian: The Cultivation Card Game,NaN,False,True,False,False,False
6,1966720,Lethal Company,NaN,Lethal Company,NaN,False,True,False,False,False
7,2115850,Last Holiday,NaN,Last Holiday,NaN,False,True,False,False,False
8,2162800,shapez 2 - Factory,NaN,shapez 2 - Factory,NaN,False,True,False,False,False
9,2186320,Ages of Conflict: World War Simulator,NaN,Ages of Conflict: World War Simulator,NaN,False,True,False,False,False


### 9-1. tags에만 있는 게임의 태그 키워드 확인

In [11]:
# tags에만 있는 appid 목록을 가져온다.
tags_only_ids = set(tags_only_not_games_df["appid"].astype(int))

tags_only_detail_df = tags_df[tags_df["appid"].astype(int).isin(tags_only_ids)].copy()

# tags 컬럼은 문자열 형태이므로, 일단 문자열 포함 여부만 확인한다.
# 실제 태그 파싱은 본 분석 단계에서 별도로 진행해도 된다.
for keyword in ["Early Access", "Free To Play", "Multiplayer", "Singleplayer", "Action", "Adventure"]:
    tags_only_detail_df[f"has_{keyword.replace(' ', '_').lower()}"] = (
        tags_only_detail_df["tags"].astype(str).str.contains(keyword, case=False, na=False)
    )

keyword_cols = [col for col in tags_only_detail_df.columns if col.startswith("has_")]
keyword_summary = tags_only_detail_df[keyword_cols].sum().reset_index()
keyword_summary.columns = ["keyword_flag", "game_count"]

display(keyword_summary)

# 대표적으로 몇 개만 확인한다.
display(
    tags_only_detail_df[["appid", "name", "developer", "price", "positive", "negative"] + keyword_cols]
    .sort_values("appid")
    .head(30)
)


,keyword_flag,game_count
0,has_early_access,12
1,has_free_to_play,2
2,has_multiplayer,6
3,has_singleplayer,8
4,has_action,7
5,has_adventure,4


,appid,name,developer,price,positive,negative,has_early_access,has_free_to_play,has_multiplayer,has_singleplayer,has_action,has_adventure
17,1049590,Eternal Return,Nimble Neuron,0,51694,13352,False,True,True,False,True,False
40,1116540,DAVIGO: VR vs. PC,Davigo Studio,2499,1018,160,True,False,True,False,True,False
31,1213300,Rebellion GODSOUL: Awakening,U-Secret Studio,1799,1645,316,True,False,False,False,False,False
77,1623730,Palworld,Pocketpair,2999,358266,22443,True,False,True,False,True,True
76,1700300,World Warfare & Economics,Okron Studio,2999,395,174,True,False,True,True,False,False
6,1948800,Yi Xian: The Cultivation Card Game,Darksun Studio,0,4531,1601,False,True,True,True,False,False
79,1966720,Lethal Company,Zeekerss,999,473422,13738,True,False,False,False,True,True
57,2115850,Last Holiday,Boomer Games s.r.o.,2999,200,248,True,False,False,True,False,True
28,2162800,shapez 2 - Factory,tobspr Games,2999,10856,205,True,False,False,True,False,False
9,2186320,Ages of Conflict: World War Simulator,JoySpark Games,699,4708,195,True,False,False,True,False,False


## 10. 리뷰 데이터에는 있는데 games에는 없는 게임 확인

In [12]:
summary_not_games_df = show_appid_difference(
    "review_summary",
    "games",
    title="review_summary에는 있지만 games에는 없는 게임"
)

histogram_not_games_df = show_appid_difference(
    "review_histogram",
    "games",
    title="review_histogram에는 있지만 games에는 없는 게임"
)

reviews_not_games_df = show_appid_difference(
    "reviews",
    "games",
    title="reviews에는 있지만 games에는 없는 게임"
)


review_summary에는 있지만 games에는 없는 게임
review_summary에는 있지만 games에는 없는 appid 수: 0


,appid,game_name,name_games,name_tags,name_review_histogram,in_games,in_tags,in_review_summary,in_review_histogram,in_reviews


review_histogram에는 있지만 games에는 없는 게임
review_histogram에는 있지만 games에는 없는 appid 수: 0


,appid,game_name,name_games,name_tags,name_review_histogram,in_games,in_tags,in_review_summary,in_review_histogram,in_reviews


reviews에는 있지만 games에는 없는 게임
reviews에는 있지만 games에는 없는 appid 수: 0


,appid,game_name,name_games,name_tags,name_review_histogram,in_games,in_tags,in_review_summary,in_review_histogram,in_reviews


## 11. summary/histogram은 있는데 reviews에는 없는 게임 확인

In [13]:
# summary와 histogram에는 있지만 reviews에는 없는 appid
summary_histogram_ids = appid_sets["review_summary"] | appid_sets["review_histogram"]
summary_histogram_but_not_reviews_ids = sorted(summary_histogram_ids - appid_sets["reviews"])

summary_histogram_but_not_reviews_df = (
    appid_ref_df[appid_ref_df["appid"].isin(summary_histogram_but_not_reviews_ids)]
    .sort_values("appid")
    .reset_index(drop=True)
)

print("review_summary 또는 review_histogram에는 있지만 reviews에는 없는 appid 수:", len(summary_histogram_but_not_reviews_df))
display(summary_histogram_but_not_reviews_df)

review_summary 또는 review_histogram에는 있지만 reviews에는 없는 appid 수: 3


,appid,game_name,name_games,name_tags,name_review_histogram,in_games,in_tags,in_review_summary,in_review_histogram,in_reviews
0,1818590,You Are A Pilot,You Are A Pilot,You Are A Pilot,You Are A Pilot,True,True,True,True,False
1,2716270,The Witch's Cauldron Prologue,The Witch's Cauldron Prologue,The Witch's Cauldron Prologue,The Witch's Cauldron Prologue,True,True,True,True,False
2,2837370,NeverGoingHome,NeverGoingHome,NeverGoingHome,NeverGoingHome,True,True,True,True,False


### 11-1. 개별 리뷰가 없는 게임의 review_summary 확인

In [14]:
missing_reviews_summary_df = (
    review_summary_df[review_summary_df["appid"].astype(int).isin(summary_histogram_but_not_reviews_ids)]
    .sort_values("appid")
    .reset_index(drop=True)
)

display(missing_reviews_summary_df)

,appid,review_score,review_score_desc,total_positive,total_negative,total_reviews
0,1818590,0,No user reviews,0,0,0
1,2716270,0,No user reviews,0,0,0
2,2837370,0,No user reviews,0,0,0


### 11-2. 개별 리뷰가 없는 게임의 review_histogram 확인


In [15]:
missing_reviews_hist_df = (
    review_histogram_df[review_histogram_df["appid"].astype(int).isin(summary_histogram_but_not_reviews_ids)]
    .sort_values(["appid", "date", "data_type"])
    .reset_index(drop=True)
)

print("histogram 행 수:", len(missing_reviews_hist_df))
display(missing_reviews_hist_df.head(100))

# appid별 합계로 요약해서 본다.
missing_reviews_hist_summary_df = (
    missing_reviews_hist_df
    .groupby("appid")
    .agg(
        game_name=("name", "first"),
        row_count=("appid", "count"),
        up_sum=("recommendations_up", "sum"),
        down_sum=("recommendations_down", "sum"),
        data_type_count=("data_type", "nunique"),
        min_date=("date", "min"),
        max_date=("date", "max"),
    )
    .reset_index()
)

display(missing_reviews_hist_summary_df)

histogram 행 수: 140


,appid,name,stratum,release_date,hist_start_date,hist_end_date,date,recommendations_up,recommendations_down,data_type
0,1818590,You Are A Pilot,Racing_mid,2024-01-06,2023-01-19,2026-04-25,2023-01-01,0,0,rollups
1,1818590,You Are A Pilot,Racing_mid,2024-01-06,2023-01-19,2026-04-25,2023-02-01,0,0,rollups
2,1818590,You Are A Pilot,Racing_mid,2024-01-06,2023-01-19,2026-04-25,2023-03-01,0,0,rollups
3,1818590,You Are A Pilot,Racing_mid,2024-01-06,2023-01-19,2026-04-25,2023-04-01,0,0,rollups
4,1818590,You Are A Pilot,Racing_mid,2024-01-06,2023-01-19,2026-04-25,2023-05-01,0,0,rollups
5,1818590,You Are A Pilot,Racing_mid,2024-01-06,2023-01-19,2026-04-25,2023-09-01,0,0,rollups
6,1818590,You Are A Pilot,Racing_mid,2024-01-06,2023-01-19,2026-04-25,2023-10-01,0,0,rollups
7,1818590,You Are A Pilot,Racing_mid,2024-01-06,2023-01-19,2026-04-25,2024-01-01,0,0,rollups
8,1818590,You Are A Pilot,Racing_mid,2024-01-06,2023-01-19,2026-04-25,2024-02-01,0,0,rollups
9,1818590,You Are A Pilot,Racing_mid,2024-01-06,2023-01-19,2026-04-25,2024-03-01,0,0,rollups


,appid,game_name,row_count,up_sum,down_sum,data_type_count,min_date,max_date
0,1818590,You Are A Pilot,55,0,0,2,2023-01-01,2026-04-25
1,2716270,The Witch's Cauldron Prologue,43,23,3,2,2023-12-01,2026-01-04
2,2837370,NeverGoingHome,42,0,0,2,2024-02-01,2026-04-17


## 12. 리뷰 테이블 간 appid 존재 패턴 정리


In [16]:
review_presence_df = appid_ref_df[
    appid_ref_df["in_review_summary"] | appid_ref_df["in_review_histogram"] | appid_ref_df["in_reviews"]
].copy()

presence_summary_df = (
    review_presence_df
    .groupby(["in_review_summary", "in_review_histogram", "in_reviews", "in_games", "in_tags"])
    .agg(
        appid_count=("appid", "count")
    )
    .reset_index()
    .sort_values("appid_count", ascending=False)
)

display(presence_summary_df)

display(review_presence_df.sort_values("appid").reset_index(drop=True))


,in_review_summary,in_review_histogram,in_reviews,in_games,in_tags,appid_count
1,True,True,True,True,True,197
0,True,True,False,True,True,3


,appid,game_name,name_games,name_tags,name_review_histogram,in_games,in_tags,in_review_summary,in_review_histogram,in_reviews
0,402160,Star Command Galaxies,Star Command Galaxies,Star Command Galaxies,Star Command Galaxies,True,True,True,True,True
1,437440,Lord of Rigel,Lord of Rigel,Lord of Rigel,Lord of Rigel,True,True,True,True,True
2,444690,TRAPPED,TRAPPED,TRAPPED,TRAPPED,True,True,True,True,True
3,571740,Golf It!,Golf It!,Golf It!,Golf It!,True,True,True,True,True
4,597920,Survivalizm - The Animal Simulator,Survivalizm - The Animal Simulator,Survivalizm - The Animal Simulator,Survivalizm - The Animal Simulator,True,True,True,True,True
5,695330,SEASON: A letter to the future,SEASON: A letter to the future,SEASON: A letter to the future,SEASON: A letter to the future,True,True,True,True,True
6,743130,MewnBase,MewnBase,MewnBase,MewnBase,True,True,True,True,True
7,754890,Firmament,Firmament,Firmament,Firmament,True,True,True,True,True
8,837350,King of the Hat,King of the Hat,King of the Hat,King of the Hat,True,True,True,True,True
9,1014260,Passageway of the Ancients,Passageway of the Ancients,Passageway of the Ancients,Passageway of the Ancients,True,True,True,True,True


## 13. 점검 결과 해석용 요약

In [17]:
print("[게임-태그 관계]")
print("games 고유 appid 수:", len(appid_sets["games"]))
print("tags 고유 appid 수 :", len(appid_sets["tags"]))
print("games에는 있고 tags에는 없는 수:", len(appid_sets["games"] - appid_sets["tags"]))
print("tags에는 있고 games에는 없는 수:", len(appid_sets["tags"] - appid_sets["games"]))

print()
print("[리뷰 표본과 games 관계]")
for name in ["review_summary", "review_histogram", "reviews"]:
    print(f"{name:16s} 고유 appid 수: {len(appid_sets[name])}")
    print(f"  - games와 매칭되는 수: {len(appid_sets[name] & appid_sets['games'])}")
    print(f"  - games에 없는 수     : {len(appid_sets[name] - appid_sets['games'])}")

print()
print("[summary/histogram은 있는데 reviews에는 없는 게임]")
print("appid 수:", len(summary_histogram_but_not_reviews_ids))
print("appid 목록:", summary_histogram_but_not_reviews_ids)

[게임-태그 관계]
games 고유 appid 수: 9692
tags 고유 appid 수 : 9706
games에는 있고 tags에는 없는 수: 0
tags에는 있고 games에는 없는 수: 14

[리뷰 표본과 games 관계]
review_summary   고유 appid 수: 200
  - games와 매칭되는 수: 200
  - games에 없는 수     : 0
review_histogram 고유 appid 수: 200
  - games와 매칭되는 수: 200
  - games에 없는 수     : 0
reviews          고유 appid 수: 197
  - games와 매칭되는 수: 197
  - games에 없는 수     : 0

[summary/histogram은 있는데 reviews에는 없는 게임]
appid 수: 3
appid 목록: [np.int64(1818590), np.int64(2716270), np.int64(2837370)]
